In [ ]:
# ===============================
# STATISTICAL DETECTORS
# ===============================

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    brier_score_loss, log_loss, classification_report
)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import scipy.stats as stats
from tqdm import tqdm

In [ ]:
# ===============================
# STEP 1: LOAD PREPARED DATA
# ===============================
print("="*70)
print("STATISTICAL DETECTOR")
print("="*70)

hc3_train = pd.read_csv("hc3_train.csv")
hc3_test = pd.read_csv("hc3_test.csv")
eli5_train = pd.read_csv("eli5_train.csv")
eli5_test = pd.read_csv("eli5_test.csv")

print(f"✅ Data loaded:")
print(f"   HC3:  Train={len(hc3_train):,} | Test={len(hc3_test):,}")
print(f"   ELI5: Train={len(eli5_train):,} | Test={len(eli5_test):,}")

In [ ]:


# ===============================
# STEP 2: LINGUISTIC FEATURE EXTRACTION
# ===============================
print("\n" + "="*70)
print("STEP 2: EXTRACTING LINGUISTIC FEATURES")
print("="*70)

class LinguisticFeatureExtractor:
    """
    Extract interpretable linguistic features for detectability analysis
    """

    def __init__(self):
        self.feature_names = []

    def extract_features(self, texts):
        """Extract features from list of texts"""
        features = []

        for text in tqdm(texts, desc="Extracting features"):
            features.append(self._extract_single(text))

        df = pd.DataFrame(features)
        self.feature_names = df.columns.tolist()
        return df

    def _extract_single(self, text):
        """Extract features from a single text"""
        words = text.split()
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]

        features = {}

        # ===== SURFACE STATISTICS =====
        features['word_count'] = len(words)
        features['char_count'] = len(text)
        features['sentence_count'] = len(sentences)
        features['avg_word_len'] = np.mean([len(w) for w in words]) if words else 0
        features['avg_sentence_len'] = len(words) / len(sentences) if sentences else 0

        # ===== LEXICAL DIVERSITY =====
        features['type_token_ratio'] = len(set(words)) / len(words) if words else 0
        features['unique_word_ratio'] = len(set(words)) / len(words) if words else 0

        # Hapax legomena (words appearing once)
        word_freq = Counter(words)
        features['hapax_ratio'] = sum(1 for c in word_freq.values() if c == 1) / len(words) if words else 0

        # ===== PUNCTUATION & FORMATTING =====
        features['comma_density'] = text.count(',') / len(words) if words else 0
        features['period_density'] = text.count('.') / len(words) if words else 0
        features['question_mark_ratio'] = text.count('?') / len(sentences) if sentences else 0
        features['exclamation_ratio'] = text.count('!') / len(sentences) if sentences else 0

        # ===== REPETITION METRICS =====
        # Bigram repetition
        bigrams = list(zip(words[:-1], words[1:]))
        features['bigram_repetition'] = 1 - (len(set(bigrams)) / len(bigrams)) if bigrams else 0

        # Trigram repetition
        trigrams = list(zip(words[:-2], words[1:-1], words[2:]))
        features['trigram_repetition'] = 1 - (len(set(trigrams)) / len(trigrams)) if trigrams else 0

        # ===== ENTROPY MEASURES =====
        # Word frequency entropy
        word_probs = np.array(list(word_freq.values())) / len(words)
        features['word_entropy'] = stats.entropy(word_probs)

        # Sentence length entropy
        sent_lens = [len(s.split()) for s in sentences]
        if len(set(sent_lens)) > 1:
            sent_len_counts = Counter(sent_lens)
            sent_len_probs = np.array(list(sent_len_counts.values())) / len(sent_lens)
            features['sentence_len_entropy'] = stats.entropy(sent_len_probs)
        else:
            features['sentence_len_entropy'] = 0

        # ===== SYNTACTIC COMPLEXITY =====
        # Sentence length variance
        features['sentence_len_variance'] = np.var(sent_lens) if sent_lens else 0
        features['sentence_len_std'] = np.std(sent_lens) if sent_lens else 0

        # ===== DISCOURSE MARKERS =====
        hedging_words = ['maybe', 'perhaps', 'possibly', 'probably', 'might', 'could', 'seem', 'appear']
        features['hedging_density'] = sum(text.lower().count(w) for w in hedging_words) / len(words) if words else 0

        certainty_words = ['definitely', 'certainly', 'obviously', 'clearly', 'always', 'never']
        features['certainty_density'] = sum(text.lower().count(w) for w in certainty_words) / len(words) if words else 0

        connector_words = ['however', 'therefore', 'moreover', 'furthermore', 'additionally', 'consequently']
        features['connector_density'] = sum(text.lower().count(w) for w in connector_words) / len(words) if words else 0

        # ===== FORMALITY MARKERS =====
        contractions = ["n't", "'ll", "'re", "'ve", "'d", "'m"]
        features['contraction_ratio'] = sum(text.count(c) for c in contractions) / len(words) if words else 0

        # ===== BURSTINESS =====
        # Measure word repetition burstiness (words appearing in clusters)
        features['burstiness'] = self._calculate_burstiness(words)

        return features

    def _calculate_burstiness(self, words):
        """Calculate burstiness metric (tendency for words to appear in bursts)"""
        if len(words) < 2:
            return 0

        word_positions = {}
        for i, word in enumerate(words):
            if word not in word_positions:
                word_positions[word] = []
            word_positions[word].append(i)

        # Calculate average gap variance
        gap_vars = []
        for positions in word_positions.values():
            if len(positions) > 1:
                gaps = np.diff(positions)
                gap_vars.append(np.var(gaps))

        return np.mean(gap_vars) if gap_vars else 0

# Extract features for all datasets
extractor = LinguisticFeatureExtractor()

print("\nExtracting features for HC3...")
hc3_train_features = extractor.extract_features(hc3_train['text'].tolist())
hc3_test_features = extractor.extract_features(hc3_test['text'].tolist())

print("\nExtracting features for ELI5...")
eli5_train_features = extractor.extract_features(eli5_train['text'].tolist())
eli5_test_features = extractor.extract_features(eli5_test['text'].tolist())

print(f"\n✅ Extracted {len(extractor.feature_names)} features:")
print(f"   {extractor.feature_names}")

# ===============================
# STEP 3: TRAIN STATISTICAL DETECTORS (with probability outputs)
# ===============================
print("\n" + "="*70)
print("STEP 3: TRAINING STATISTICAL DETECTORS")
print("="*70)

# Encode labels: human=0, llm=1 (for detectability score)
def encode_labels(labels):
    return (labels == 'llm').astype(int)

y_hc3_train = encode_labels(hc3_train['label'])
y_hc3_test = encode_labels(hc3_test['label'])
y_eli5_train = encode_labels(eli5_train['label'])
y_eli5_test = encode_labels(eli5_test['label'])

# Initialize detectors
detectors = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42)
}

# Storage for results
results = {}

# Train each detector on both datasets
for detector_name, detector in detectors.items():
    print(f"\n{'='*70}")
    print(f"Training: {detector_name}")
    print(f"{'='*70}")

    results[detector_name] = {}

    # ===== TRAIN ON HC3, TEST ON HC3 (in-distribution) =====
    print("\n[HC3 → HC3] In-distribution")
    detector.fit(hc3_train_features, y_hc3_train)

    # Get detectability scores (probability of being LLM)
    detectability_scores_hc3 = detector.predict_proba(hc3_test_features)[:, 1]

    results[detector_name]['hc3_to_hc3'] = {
        'y_true': y_hc3_test,
        'y_pred': (detectability_scores_hc3 > 0.5).astype(int),
        'detectability_scores': detectability_scores_hc3,
        'roc_auc': roc_auc_score(y_hc3_test, detectability_scores_hc3),
        'brier_score': brier_score_loss(y_hc3_test, detectability_scores_hc3),
        'log_loss': log_loss(y_hc3_test, detectability_scores_hc3)
    }

    print(f"   ROC-AUC: {results[detector_name]['hc3_to_hc3']['roc_auc']:.4f}")
    print(f"   Brier Score: {results[detector_name]['hc3_to_hc3']['brier_score']:.4f}")

    # ===== TRAIN ON HC3, TEST ON ELI5 (cross-distribution) =====
    print("\n[HC3 → ELI5] Cross-distribution")
    detectability_scores_eli5 = detector.predict_proba(eli5_test_features)[:, 1]

    results[detector_name]['hc3_to_eli5'] = {
        'y_true': y_eli5_test,
        'y_pred': (detectability_scores_eli5 > 0.5).astype(int),
        'detectability_scores': detectability_scores_eli5,
        'roc_auc': roc_auc_score(y_eli5_test, detectability_scores_eli5),
        'brier_score': brier_score_loss(y_eli5_test, detectability_scores_eli5),
        'log_loss': log_loss(y_eli5_test, detectability_scores_eli5)
    }

    print(f"   ROC-AUC: {results[detector_name]['hc3_to_eli5']['roc_auc']:.4f}")
    print(f"   Brier Score: {results[detector_name]['hc3_to_eli5']['brier_score']:.4f}")

    # ===== TRAIN ON ELI5, TEST ON ELI5 (in-distribution) =====
    print("\n[ELI5 → ELI5] In-distribution")
    detector.fit(eli5_train_features, y_eli5_train)
    detectability_scores_eli5_in = detector.predict_proba(eli5_test_features)[:, 1]

    results[detector_name]['eli5_to_eli5'] = {
        'y_true': y_eli5_test,
        'y_pred': (detectability_scores_eli5_in > 0.5).astype(int),
        'detectability_scores': detectability_scores_eli5_in,
        'roc_auc': roc_auc_score(y_eli5_test, detectability_scores_eli5_in),
        'brier_score': brier_score_loss(y_eli5_test, detectability_scores_eli5_in),
        'log_loss': log_loss(y_eli5_test, detectability_scores_eli5_in)
    }

    print(f"   ROC-AUC: {results[detector_name]['eli5_to_eli5']['roc_auc']:.4f}")
    print(f"   Brier Score: {results[detector_name]['eli5_to_eli5']['brier_score']:.4f}")

    # ===== TRAIN ON ELI5, TEST ON HC3 (cross-distribution) =====
    print("\n[ELI5 → HC3] Cross-distribution")
    detectability_scores_hc3_cross = detector.predict_proba(hc3_test_features)[:, 1]

    results[detector_name]['eli5_to_hc3'] = {
        'y_true': y_hc3_test,
        'y_pred': (detectability_scores_hc3_cross > 0.5).astype(int),
        'detectability_scores': detectability_scores_hc3_cross,
        'roc_auc': roc_auc_score(y_hc3_test, detectability_scores_hc3_cross),
        'brier_score': brier_score_loss(y_hc3_test, detectability_scores_hc3_cross),
        'log_loss': log_loss(y_hc3_test, detectability_scores_hc3_cross)
    }

    print(f"   ROC-AUC: {results[detector_name]['eli5_to_hc3']['roc_auc']:.4f}")
    print(f"   Brier Score: {results[detector_name]['eli5_to_hc3']['brier_score']:.4f}")

    for eval_name, eval_data in results[detector_name].items():
        safe_name = detector_name.replace(" ", "_").replace("(", "").replace(")", "")
        np.save(
            f"stat_scores_{safe_name}_{eval_name}.npy",
            {
                "y_true":  eval_data["y_true"],
                "y_score": eval_data["detectability_scores"]
            }
        )



In [ ]:
# ===============================
# STEP 4: DETECTABILITY SCORE DISTRIBUTIONS
# ===============================
print("\n" + "="*70)
print("STEP 4: ANALYZING DETECTABILITY SCORE DISTRIBUTIONS")
print("="*70)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle('Detectability Score Distributions (Continuous Signal)', fontsize=16)

for i, (detector_name, detector_results) in enumerate(results.items()):
    for j, (eval_name, eval_data) in enumerate(detector_results.items()):
        ax = axes[i, j]

        # Separate scores by true label
        human_scores = eval_data['detectability_scores'][eval_data['y_true'] == 0]
        llm_scores = eval_data['detectability_scores'][eval_data['y_true'] == 1]

        # Plot distributions
        ax.hist(human_scores, bins=30, alpha=0.6, label='Human', color='blue', range=(0, 1))
        ax.hist(llm_scores, bins=30, alpha=0.6, label='LLM', color='red', range=(0, 1))

        ax.set_xlabel('Detectability Score (0=Human, 1=LLM)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{detector_name}\n{eval_name}')
        ax.legend()
        ax.axvline(0.5, color='black', linestyle='--', alpha=0.3)
        ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

# ===============================
# STEP 5: CALIBRATION CURVES
# ===============================
print("\n" + "="*70)
print("STEP 5: CALIBRATION ANALYSIS")
print("="*70)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle('Calibration Curves (Are confidence scores reliable?)', fontsize=16)

for i, (detector_name, detector_results) in enumerate(results.items()):
    for j, (eval_name, eval_data) in enumerate(detector_results.items()):
        ax = axes[i, j]

        # Compute calibration curve
        fraction_of_positives, mean_predicted_value = calibration_curve(
            eval_data['y_true'],
            eval_data['detectability_scores'],
            n_bins=10,
            strategy='uniform'
        )

        # Plot
        ax.plot(mean_predicted_value, fraction_of_positives, 's-', label='Detector')
        ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
        ax.set_xlabel('Mean Predicted Detectability')
        ax.set_ylabel('Fraction of LLM Text')
        ax.set_title(f'{detector_name}\n{eval_name}')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ===============================
# STEP 6: SUMMARY TABLE
# ===============================
print("\n" + "="*70)
print("STEP 6: PERFORMANCE SUMMARY")
print("="*70)

summary_rows = []
for detector_name, detector_results in results.items():
    for eval_name, eval_data in detector_results.items():
        summary_rows.append({
            'Detector': detector_name,
            'Evaluation': eval_name,
            'ROC-AUC': eval_data['roc_auc'],
            'Brier Score': eval_data['brier_score'],
            'Log Loss': eval_data['log_loss'],
            'Mean Human Score': eval_data['detectability_scores'][eval_data['y_true'] == 0].mean(),
            'Mean LLM Score': eval_data['detectability_scores'][eval_data['y_true'] == 1].mean(),
            'Score Overlap': None
        })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# Save results
summary_df.to_csv("detector_family1_results.csv", index=False)
print("\n✅ Saved: detector_family1_results.csv")